# The Infosiast — Post-Quantum Cryptography (NIST FIPS 203 & 204) Packet & Latency Benchmark
**Prepared by [Kshitij Gupta](https://www.linkedin.com/in/kshitij-gupta-855535134/) for [The Infosiast](https://theinfosiast.com/)**

Accompanies: [Post-Quantum Cryptography: Zero-Trust Migration Blueprint](https://theinfosiast.com/post-quantum-cryptography-zero-trust-migration-blueprint/)

This notebook compares payload sizes, Ethernet MTU fragmentation thresholds (`1500` bytes), and cryptographic handshake overhead across classical `X25519`, hybrid `X25519MLKEM768` (FIPS 203), and `ML-DSA-65` (FIPS 204) certificate chains.


### Step 1: NIST FIPS 203 (ML-KEM) & FIPS 204 (ML-DSA) Wire-Size & MTU Fragmentation Analysis


In [ ]:
import math
import pandas as pd

ETHERNET_MSS = 1460  # Standard TCP Maximum Segment Size over 1500-byte Ethernet MTU

primitives = [
    {'Algorithm': 'X25519 (Classical ECDHE)', 'Standard': 'RFC 7748', 'Public Key (B)': 32, 'Ciphertext / Sig (B)': 32},
    {'Algorithm': 'X25519MLKEM768 (Hybrid KEX)', 'Standard': 'NIST FIPS 203 + X25519', 'Public Key (B)': 1216, 'Ciphertext / Sig (B)': 1120},
    {'Algorithm': 'ML-KEM-1024 (PQC KEX Level 5)', 'Standard': 'NIST FIPS 203', 'Public Key (B)': 1568, 'Ciphertext / Sig (B)': 1568},
    {'Algorithm': 'ECDSA P-256 (Classical Auth)', 'Standard': 'FIPS 186-5', 'Public Key (B)': 64, 'Ciphertext / Sig (B)': 64},
    {'Algorithm': 'ML-DSA-65 (PQC Signature)', 'Standard': 'NIST FIPS 204', 'Public Key (B)': 1952, 'Ciphertext / Sig (B)': 3309},
    {'Algorithm': 'SLH-DSA-SHA2-128f (Hash Sig)', 'Standard': 'NIST FIPS 205', 'Public Key (B)': 32, 'Ciphertext / Sig (B)': 17088},
]

for row in primitives:
    handshake_bytes = 320 + row['Public Key (B)'] + row['Ciphertext / Sig (B)']
    row['Total Wire Overhead (B)'] = handshake_bytes
    row['TCP Segments (1460B MSS)'] = math.ceil(handshake_bytes / ETHERNET_MSS)

df = pd.DataFrame(primitives)
print(df.to_string(index=False))


### Step 2: Live TLS 1.3 Cipher & Ephemeral Key Exchange Verification


In [ ]:
import ssl, socket

hostname = 'theinfosiast.com'
ctx = ssl.create_default_context()
with socket.create_connection((hostname, 443), timeout=10) as sock:
    with ctx.wrap_socket(sock, server_hostname=hostname) as ssock:
        print('Connected Host :', hostname)
        print('TLS Version    :', ssock.version())
        print('Negotiated Suite:', ssock.cipher())
